<a href="https://colab.research.google.com/github/Shobby101/DNA-RNA-Sequence-Analyzer/blob/main/Skills_Gap_Pattern_Discovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving demand_processed.csv to demand_processed.csv


In [ ]:
!pip install mlxtend gensim pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 28.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# PART 1: APRIORI ALGORITHM (Skill Bundles)
# ============================================================

class AprioriSkillBundles:
    """
    Finds frequently co-occurring skills in job postings
    Optimized for memory: uses transaction encoding with limit on skills
    """

    def __init__(self, demand_processed_file, min_support=0.02, min_confidence=0.5, min_lift=1.2):
        """
        Args:
            demand_processed_file: Path to demand_processed.csv
            min_support: Minimum support (default 0.02 = 2% of jobs)
            min_confidence: Minimum confidence (0.5 = 50%)
            min_lift: Minimum lift (1.2 = positive correlation)
        """
        self.filepath = demand_processed_file
        self.min_support = min_support
        self.min_confidence = min_confidence
        self.min_lift = min_lift
        self.rules = None
        self.frequent_itemsets = None

    def load_and_prepare_transactions(self, max_transactions=None, max_skills_per_transaction=15):
        """
        Load skills from processed demand data and prepare transactions
        Limits data to fit in memory
        """
        print("\n" + "=" * 60)
        print("APRIORI: Preparing skill transactions")
        print("=" * 60)

        # Load processed demand data
        df = pd.read_csv(self.filepath)
        print(f"Loaded {len(df)} job postings")

        # Get skills column
        skills_col = None
        for col in ['normalized_skills', 'required_skills', 'skills']:
            if col in df.columns:
                skills_col = col
                break

        if skills_col is None:
            raise ValueError("No skills column found in demand file")

        # Parse skills
        transactions = []
        for idx, row in df.iterrows():
            skills = row[skills_col]
            if isinstance(skills, str):
                try:
                    skills = ast.literal_eval(skills)
                except:
                    # Fallback parsing
                    skills = [s.strip() for s in skills.strip('[]').replace("'", "").split(',') if s.strip()]
            if not isinstance(skills, list):
                skills = []

            # Limit skills per transaction to reduce memory
            if len(skills) > max_skills_per_transaction:
                skills = skills[:max_skills_per_transaction]

            if skills:
                transactions.append(skills)

            # Stop early if max_transactions set (for testing)
            if max_transactions and len(transactions) >= max_transactions:
                break

        print(f"Prepared {len(transactions)} transactions")
        print(f"Average skills per transaction: {np.mean([len(t) for t in transactions]):.1f}")

        # Filter to skills that appear frequently enough (reduce dimensions)
        from collections import Counter
        skill_counts = Counter([s for t in transactions for s in t])
        min_freq = max(2, int(len(transactions) * self.min_support))
        frequent_skills = {skill for skill, count in skill_counts.items() if count >= min_freq}

        # Filter transactions to only frequent skills
        filtered_transactions = [[s for s in t if s in frequent_skills] for t in transactions]
        filtered_transactions = [t for t in filtered_transactions if len(t) >= 2]

        print(f"Kept {len(frequent_skills)} frequent skills (appear in ≥ {min_freq} jobs)")
        print(f"Remaining transactions: {len(filtered_transactions)}")

        return filtered_transactions

    def run_apriori(self, transactions, max_itemset_size=4):
        """
        Run Apriori algorithm using mlxtend (must install)
        If not installed, falls back to manual implementation
        """
        try:
            from mlxtend.frequent_patterns import apriori, association_rules
            from mlxtend.preprocessing import TransactionEncoder

            print("\n🔍 Encoding transactions...")
            te = TransactionEncoder()
            te_ary = te.fit(transactions).transform(transactions)
            df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

            print(f"Encoded shape: {df_encoded.shape}")

            # Find frequent itemsets
            print("🔍 Finding frequent itemsets (this may take a moment)...")
            frequent_itemsets = apriori(
                df_encoded,
                min_support=self.min_support,
                use_colnames=True,
                max_len=max_itemset_size
            )

            if len(frequent_itemsets) == 0:
                print("No frequent itemsets found. Try lowering min_support.")
                return None, None

            print(f"Found {len(frequent_itemsets)} frequent itemsets")

            # Generate rules
            rules = association_rules(
                frequent_itemsets,
                metric="lift",
                min_threshold=self.min_lift
            )

            # Filter by confidence
            rules = rules[rules['confidence'] >= self.min_confidence]
            rules = rules.sort_values('lift', ascending=False)

            print(f"Generated {len(rules)} association rules")

            # Convert frozenset to strings for readability
            rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
            rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

            self.frequent_itemsets = frequent_itemsets
            self.rules = rules
            return rules, frequent_itemsets

        except ImportError:
            print("⚠️ mlxtend not installed. Installing...")
            import subprocess
            subprocess.check_call(['pip', 'install', 'mlxtend'])
            # Retry
            from mlxtend.frequent_patterns import apriori, association_rules
            from mlxtend.preprocessing import TransactionEncoder
            return self.run_apriori(transactions, max_itemset_size)

    def get_top_bundles(self, n=15):
        """Get top skill bundles by lift"""
        if self.rules is None:
            print("Run run_apriori() first")
            return None
        return self.rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(n)

    def save_rules(self, filename='skill_bundles.csv'):
        if self.rules is not None:
            self.rules.to_csv(filename, index=False)
            print(f"✓ Saved: {filename}")
        else:
            print("No rules to save")


# ============================================================
# PART 2: LDA TOPIC MODELING (Latent Job Roles)
# ============================================================

class LDATopicDiscovery:
    """
    Discovers latent job roles from job descriptions using LDA
    Optimized for 8GB RAM: limits vocabulary, uses gensim with chunking
    """

    def __init__(self, demand_processed_file, n_topics=10, passes=5):
        """
        Args:
            demand_processed_file: Path to demand_processed.csv
            n_topics: Number of topics to discover (reduce for less RAM)
            passes: Number of training passes (reduce if slow)
        """
        self.filepath = demand_processed_file
        self.n_topics = min(n_topics, 12)  # Cap at 12 for 8GB RAM
        self.passes = passes
        self.lda_model = None
        self.topics = None

    def load_and_prepare_text(self, max_docs=None, max_words_per_doc=500):
        """
        Load job descriptions, clean, and prepare for LDA
        Limits data to fit memory
        """
        print("\n" + "=" * 60)
        print("LDA: Preparing job descriptions for topic modeling")
        print("=" * 60)

        df = pd.read_csv(self.filepath)
        print(f"Loaded {len(df)} job postings")

        # Get description column
        desc_col = None
        for col in ['description', 'job_description', 'text']:
            if col in df.columns:
                desc_col = col
                break

        if desc_col is None:
            print("No description column found. Using job titles instead.")
            desc_col = 'job_title'

        # Also try to use normalized_title if available
        title_col = 'normalized_title' if 'normalized_title' in df.columns else None

        # Prepare documents
        documents = []
        for idx, row in df.iterrows():
            # Combine title and description if available
            text = ""
            if title_col and pd.notna(row[title_col]):
                text += str(row[title_col]) + ". "
            if pd.notna(row[desc_col]):
                text += str(row[desc_col])[:max_words_per_doc]

            if text.strip():
                documents.append(text)

            if max_docs and len(documents) >= max_docs:
                break

        print(f"Prepared {len(documents)} documents")
        return documents

    def clean_and_tokenize(self, documents):
        """
        Clean text and tokenize for LDA
        Uses simple tokenization to save memory
        """
        import re

        # Custom stop words for tech job descriptions
        stop_words = {
            'the', 'and', 'for', 'with', 'this', 'that', 'are', 'will', 'have', 'from',
            'they', 'their', 'your', 'our', 'can', 'all', 'about', 'also', 'any', 'been',
            'but', 'has', 'was', 'were', 'experience', 'ability', 'knowledge', 'skills',
            'strong', 'work', 'team', 'development', 'technical', 'working', 'using',
            'job', 'position', 'candidate', 'looking', 'seeking', 'require', 'required',
            'must', 'should', 'would', 'could', 'within', 'without', 'after', 'before',
            'under', 'over', 'through', 'during', 'between', 'among', 'throughout'
        }

        tokenized_docs = []

        for doc in documents:
            # Lowercase
            doc = doc.lower()
            # Remove punctuation
            doc = re.sub(r'[^a-zA-Z\s]', ' ', doc)
            # Split
            tokens = doc.split()
            # Remove stop words and short tokens
            tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
            # Keep only words that appear in skill taxonomy (optional filter)
            tokenized_docs.append(tokens)

        print(f"Tokenized {len(tokenized_docs)} documents")
        print(f"Average tokens per doc: {np.mean([len(t) for t in tokenized_docs]):.1f}")

        return tokenized_docs

    def run_lda(self, tokenized_docs):
        """
        Run LDA using gensim (optimized for memory)
        """
        try:
            import gensim
            from gensim import corpora, models

            print("\n🔍 Creating dictionary and corpus...")

            # Create dictionary (filter extremes to reduce memory)
            dictionary = corpora.Dictionary(tokenized_docs)
            dictionary.filter_extremes(no_below=5, no_above=0.5, keep_n=2000)  # Limit vocab to 2000

            print(f"Dictionary size: {len(dictionary)} terms")

            # Create bag-of-words corpus
            corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]
            print(f"Corpus size: {len(corpus)} documents")

            # Run LDA with optimized settings for 8GB RAM
            print(f"🔍 Running LDA with {self.n_topics} topics (this may take a few minutes)...")

            lda_model = gensim.models.LdaModel(
                corpus=corpus,
                id2word=dictionary,
                num_topics=self.n_topics,
                random_state=42,
                passes=self.passes,
                iterations=50,          # Reduced from 100
                chunksize=1000,         # Process in small chunks
                alpha='auto',
                eta='auto',
                per_word_topics=False   # Saves memory
            )

            self.lda_model = lda_model
            self.dictionary = dictionary
            self.corpus = corpus

            # Extract topics
            topics = []
            for topic_id in range(self.n_topics):
                top_words = lda_model.show_topic(topic_id, topn=8)
                topics.append({
                    'topic_id': topic_id,
                    'keywords': [word for word, prob in top_words],
                    'topic_string': ', '.join([word for word, prob in top_words])
                })

            self.topics = topics

            print(f"\n✅ LDA complete! Discovered {len(topics)} topics")
            return lda_model, topics

        except ImportError:
            print("⚠️ gensim not installed. Installing...")
            import subprocess
            subprocess.check_call(['pip', 'install', 'gensim'])
            import gensim
            from gensim import corpora, models
            return self.run_lda(tokenized_docs)

    def assign_dominant_topics(self):
        """Assign each job posting to its dominant topic"""
        if self.lda_model is None:
            print("Run run_lda() first")
            return None

        topic_assignments = []
        for idx, bow in enumerate(self.corpus):
            topic_probs = self.lda_model.get_document_topics(bow)
            if topic_probs:
                dominant_topic = max(topic_probs, key=lambda x: x[1])[0]
                topic_assignments.append({
                    'doc_id': idx,
                    'dominant_topic': dominant_topic,
                    'confidence': max(topic_probs, key=lambda x: x[1])[1]
                })

        df_topics = pd.DataFrame(topic_assignments)
        return df_topics

    def save_topics(self, filename='lda_topics.csv'):
        if self.topics is not None:
            df = pd.DataFrame(self.topics)
            df.to_csv(filename, index=False)
            print(f"✓ Saved: {filename}")
        else:
            print("No topics to save")

    def print_topics(self):
        """Print discovered topics in readable format"""
        if self.topics is None:
            print("Run run_lda() first")
            return

        print("\n" + "=" * 60)
        print("DISCOVERED JOB ROLES (LDA Topics)")
        print("=" * 60)

        for topic in self.topics:
            print(f"\n📌 Topic {topic['topic_id']}:")
            print(f"   Keywords: {topic['topic_string']}")

            # Try to infer role name
            keywords_lower = topic['topic_string'].lower()
            if 'python' in keywords_lower or 'django' in keywords_lower:
                role = "Python/Backend Developer"
            elif 'react' in keywords_lower or 'javascript' in keywords_lower:
                role = "Frontend/React Developer"
            elif 'aws' in keywords_lower or 'docker' in keywords_lower or 'cloud' in keywords_lower:
                role = "DevOps/Cloud Engineer"
            elif 'data' in keywords_lower or 'machine' in keywords_lower or 'sql' in keywords_lower:
                role = "Data Scientist/Analyst"
            elif 'java' in keywords_lower or 'spring' in keywords_lower:
                role = "Java Developer"
            elif 'mobile' in keywords_lower or 'android' in keywords_lower or 'ios' in keywords_lower:
                role = "Mobile Developer"
            else:
                role = "General Software Developer"

            print(f"   → Inferred Role: {role}")


# ============================================================
# PART 3: MAIN EXECUTION (Optimized for 8GB RAM)
# ============================================================

def run_pattern_discovery(demand_file='demand_processed.csv'):
    """
    Run both Apriori and LDA with memory-optimized settings
    """
    print("\n" + "=" * 70)
    print("PATTERN DISCOVERY FOR SKILLS GAP ANALYSIS")
    print("Apriori (Skill Bundles) + LDA (Job Roles)")
    print("Optimized for 8GB RAM")
    print("=" * 70)

    # ========================================================
    # APRIORI: Skill Bundles
    # ========================================================
    print("\n" + "🔧" * 35)
    print("PHASE 1: Apriori Algorithm - Finding Skill Bundles")
    print("🔧" * 35)

    # Initialize with conservative thresholds for 8GB RAM
    apriori = AprioriSkillBundles(
        demand_file,
        min_support=0.03,      # Higher support = fewer itemsets = less memory
        min_confidence=0.5,
        min_lift=1.2
    )

    # Limit to 1000 transactions for testing, but you can remove limit
    transactions = apriori.load_and_prepare_transactions(max_transactions=None, max_skills_per_transaction=12)

    if len(transactions) > 0:
        rules, itemsets = apriori.run_apriori(transactions, max_itemset_size=3)  # Limit itemset size to 3 for memory
        if rules is not None and len(rules) > 0:
            top_bundles = apriori.get_top_bundles(15)
            print("\n" + "=" * 60)
            print("TOP 15 SKILL BUNDLES (by Lift)")
            print("=" * 60)
            print(f"{'If you have...':<30} {'You likely also need...':<30} {'Lift':<10}")
            print("-" * 70)
            for idx, row in top_bundles.iterrows():
                antecedents = row['antecedents'][:25] + ".." if len(row['antecedents']) > 25 else row['antecedents']
                consequents = row['consequents'][:25] + ".." if len(row['consequents']) > 25 else row['consequents']
                print(f"{antecedents:<30} {consequents:<30} {row['lift']:.2f}")

            apriori.save_rules('skill_bundles_apriori.csv')
        else:
            print("No rules found. Try lowering min_support to 0.02")
    else:
        print("No transactions prepared. Check your data.")

    # ========================================================
    # LDA: Topic Modeling
    # ========================================================
    print("\n\n" + "🧠" * 35)
    print("PHASE 2: LDA Topic Modeling - Discovering Latent Job Roles")
    print("🧠" * 35)

    # Use fewer topics and passes for 8GB RAM
    lda = LDATopicDiscovery(demand_file, n_topics=8, passes=3)  # Reduced for memory

    # Limit documents to 1500 for memory
    documents = lda.load_and_prepare_text(max_docs=1500, max_words_per_doc=400)

    if len(documents) > 50:
        tokenized = lda.clean_and_tokenize(documents)
        model, topics = lda.run_lda(tokenized)

        if model:
            lda.print_topics()
            lda.save_topics('lda_topics.csv')

            # Optional: Assign dominant topics
            topic_assignments = lda.assign_dominant_topics()
            if topic_assignments is not None:
                topic_assignments.to_csv('job_topic_assignments.csv', index=False)
                print("\n✓ Saved: job_topic_assignments.csv")
    else:
        print("Not enough documents for LDA. Need at least 50.")

    print("\n" + "=" * 70)
    print("✅ PATTERN DISCOVERY COMPLETE!")
    print("   Files created:")
    print("   - skill_bundles_apriori.csv")
    print("   - lda_topics.csv")
    print("   - job_topic_assignments.csv")
    print("=" * 70)


if __name__ == "__main__":
    import os

    # ❌ Remove this line for Colab (causes error)
    # os.chdir(os.path.dirname(os.path.abspath(__file__)))

    # ✅ Just use the current working directory (already /content/)
    print(f"Working directory: {os.getcwd()}")

    # Check if demand_processed.csv exists
    if not os.path.exists('demand_processed.csv'):
        print("❌ demand_processed.csv not found!")
        print("   Please upload the file using files.upload() first.")
        exit(1)

    # Run pattern discovery (ensure function name matches: run_pattern_discovery)
    run_pattern_discovery('demand_processed.csv')

Working directory: /content

PATTERN DISCOVERY FOR SKILLS GAP ANALYSIS
Apriori (Skill Bundles) + LDA (Job Roles)
Optimized for 8GB RAM

🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧
PHASE 1: Apriori Algorithm - Finding Skill Bundles
🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧🔧

APRIORI: Preparing skill transactions
Loaded 2000 job postings
Prepared 2000 transactions
Average skills per transaction: 10.5
Kept 92 frequent skills (appear in ≥ 60 jobs)
Remaining transactions: 2000

🔍 Encoding transactions...
Encoded shape: (2000, 92)
🔍 Finding frequent itemsets (this may take a moment)...
Found 2314 frequent itemsets


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Generated 5468 association rules

TOP 15 SKILL BUNDLES (by Lift)
If you have...                 You likely also need...        Lift      
----------------------------------------------------------------------
Firewalls                      Network Security               24.60
Network Security               Firewalls                      24.60
Information Architecture,..    Prototyping                    23.71
Prototyping                    Information Architecture,..    23.71
Teamwork, Adobe Xd             Prototyping                    23.70
Teamwork, Adobe Xd             Usability Testing              23.70
Usability Testing              Teamwork, Adobe Xd             23.70
Prototyping                    Teamwork, Adobe Xd             23.70
Visual Design                  Wireframing, Teamwork          23.62
Teamwork, Usability Testi..    Visual Design                  23.62
Visual Design                  Teamwork, Usability Testi..    23.62
Wireframing, Teamwork          Visual Desig

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Prepared 1500 documents
Tokenized 1500 documents
Average tokens per doc: 30.6


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag


✅ LDA complete! Discovered 8 topics

DISCOVERED JOB ROLES (LDA Topics)

📌 Topic 0:
   Keywords: data, analyst, designer, engineer, learning, python, business, intelligence
   → Inferred Role: Python/Backend Developer

📌 Topic 1:
   Keywords: react, css, design, database, tailwind, figma, software, responsive
   → Inferred Role: Frontend/React Developer

📌 Topic 2:
   Keywords: product, engineer, market, build, african, hybrid, help, talented
   → Inferred Role: General Software Developer

📌 Topic 3:
   Keywords: hybrid, talented, products, role, title, company, location, build
   → Inferred Role: General Software Developer

📌 Topic 4:
   Keywords: problem, engineer, mobile, solving, git, react, software, platform
   → Inferred Role: Frontend/React Developer

📌 Topic 5:
   Keywords: software, java, python, javascript, object, oriented, programming, algorithms
   → Inferred Role: Python/Backend Developer

📌 Topic 6:
   Keywords: stack, full, problem, node, software, git, docker, api
   

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



✓ Saved: job_topic_assignments.csv

✅ PATTERN DISCOVERY COMPLETE!
   Files created:
   - skill_bundles_apriori.csv
   - lda_topics.csv
   - job_topic_assignments.csv


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
from google.colab import files
files.download('skill_bundles_apriori.csv')
files.download('lda_topics.csv')
files.download('job_topic_assignments.csv')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag